# HW3 - PEFT

In this notebook, we will fine-tune the GPT2 model on the [WikiText](https://huggingface.co/datasets/Salesforce/wikitext#wikitext-2-v1) dataset using different fine-tuning methodologies.

Parameter-Efficient Fine-Tuning (PEFT) is a technique that enables the adaptation of large pre-trained models to specific tasks while modifying only a small subset of their parameters, significantly reducing computational and memory costs. Instead of updating all model parameters, PEFT methods, such as LoRA (Low-Rank Adaptation), Adapter layers, and Prefix-Tuning, introduce lightweight trainable modules that are inserted into the model or modify activations in a structured way. This approach retains the general knowledge of the base model while efficiently adapting to new tasks, making it particularly useful for fine-tuning large-scale models like LLMs and vision-language models on resource-constrained hardware.

## Install required libraries

In [1]:
!pip install datasets

## Import required libraries

In [2]:
import gc
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, PrefixTuningConfig, get_peft_model, PeftModel

## Setup

In [3]:
gpt_2_medium_model_name = "openai-community/gpt2-medium"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(gpt_2_medium_model_name)
tokenizer.pad_token = tokenizer.eos_token

# Tokenize the dataset
def tokenizing_preprocess(examples):
    inputs =  tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)
    inputs['labels'] = inputs['input_ids'].copy()
    return inputs


# Define training arguments
training_args = TrainingArguments(
    output_dir='./gpt2',
    eval_strategy='no',
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    report_to="none"
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load dataset (5 pt)

In [4]:
# TODO: Load the wikitext-2-v1 version of wikitext
dataset = load_dataset("wikitext", "wikitext-2-v1")

In [5]:
# TODO Select 1000 data for train and 500 data for validation
train_data = dataset["train"].select(range(1000))
eval_data = dataset["validation"].select(range(500))

# Apply tokenization preprocess on datasets
train_dataset = train_data.map(tokenizing_preprocess, batched=True)
eval_dataset = eval_data.map(tokenizing_preprocess, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## Full Fine-Tuning (5 pt)

In [6]:
# Load the model
ff_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)

In [7]:
# Initialize Trainer
trainer = Trainer(
    model=ff_model,
    args=training_args,
    train_dataset=train_dataset,
)

In [8]:
# Zero-Shot evaluation of model

# TODO: Evaluate model on eval_dataset
eval_output = trainer.evaluate(eval_dataset=eval_dataset)


print(f"eval_loss = {eval_output['eval_loss']:.4f}")

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


eval_loss = 6.9568


In [9]:
# TODO: Get reserved memory from cuda
gpu_memory_before = torch.cuda.memory_reserved()
# TODO: Train the model using trainer
train_output = trainer.train()
# TODO: Get reserved memory from cuda
gpu_memory_after = torch.cuda.memory_reserved()

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")
print(f"GPU memory used: {(gpu_memory_after - gpu_memory_before) / (1024 ** 3):.4f} GB")

Step,Training Loss


Training time: 120.9722 seconds
GPU memory used: 5542772736.0000 bytes
GPU memory used: 5.1621 GB


In [10]:
# TODO: Evaluate model on eval_dataset
eval_output = trainer.evaluate(eval_dataset=eval_dataset)

print(f"eval_loss = {eval_output['eval_loss']:.4f}")

eval_loss = 1.0830


In [11]:
# Delete the model
del ff_model
del trainer

In [12]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close to zero(~0.2))
gc.collect()
torch.cuda.empty_cache()

## Prefix Tuning (20 pt)

TODO: Explain about Prefix Tuning briefly

Prefix Tuning is a Parameter-Efficient Fine-Tuning (PEFT) technique that adapts a pre-trained language model to new tasks by prepending trainable 'prefix' tokens to the model’s input.

Instead of updating all model parameters, it keeps the original model frozen and only optimizes a small set of continuous vectors that represent task-specific information. These vectors act as pseudo-inputs and modify the attention mechanism by altering key and value projections.

This method is efficient in terms of both memory and computation, and is particularly useful when we need to adapt large models with limited resources or data.
"""

In [13]:
from transformers import AutoModel
prefix_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)

In [14]:
# TODO: Define your LoRA configuration using PrefixTuningConfig class from peft library
#       Set task_type to CAUSAL_LM

prefix_config = PrefixTuningConfig(
    task_type="CAUSAL_LM",
    num_virtual_tokens=10,
    encoder_hidden_size=128
)


# TODO: Wrraped the GPT2LMHeadModel with above prefix config using get_peft_model function
prefix_model = get_peft_model(prefix_model, prefix_config)

# TODO: Print number of trainable parameters
prefix_model.print_trainable_parameters()

trainable params: 491,520 || all params: 355,314,688 || trainable%: 0.1383


In [15]:
prefix_model

PeftModelForCausalLM(
  (base_model): GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(50257, 1024)
      (wpe): Embedding(1024, 1024)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-23): 24 x GPT2Block(
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D(nf=3072, nx=1024)
            (c_proj): Conv1D(nf=1024, nx=1024)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D(nf=4096, nx=1024)
            (c_proj): Conv1D(nf=1024, nx=4096)
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    )
    (lm_head): Linear(in_fea

In [16]:
# Initialize Trainer
trainer = Trainer(
    model=prefix_model,
    args=training_args,
    train_dataset=train_dataset,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [17]:
# TODO: Get reserved memory from cuda
gpu_memory_before = torch.cuda.memory_reserved()
# TODO: Train the model
train_output = trainer.train()
# TODO: Get reserved memory from cuda
gpu_memory_after = torch.cuda.memory_reserved()

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")
print(f"GPU memory used: {(gpu_memory_after - gpu_memory_before) / (1024 ** 3):.4f} GB")

Step,Training Loss


Training time: 65.7186 seconds
GPU memory used: 1566572544.0000 bytes
GPU memory used: 1.4590 GB


In [18]:
# TODO: Evaluate model on eval_dataset
eval_output = trainer.evaluate(eval_dataset=eval_dataset)

print(f"eval_loss = {eval_output['eval_loss']:.4f}")

eval_loss = 9.4654


In [19]:
# Delete the model
del prefix_model
del trainer

In [20]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close or less than 1.5Gb)
gc.collect()
torch.cuda.empty_cache()

## Fine-Tuning by LoRA (Low-Rank Adaptation) (40 pt)

TODO: Explain about LoRA (Low-Rank Adaptation) briefly

LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique that reduces the number of trainable parameters by decomposing weight updates into low-rank matrices.

Instead of updating the full weight matrices of a pre-trained model during fine-tuning, LoRA keeps the original weights frozen and injects a pair of trainable low-rank matrices (A and B) into the architecture. These matrices approximate the full-rank weight update while being much smaller in size.

This method significantly reduces memory and compute cost while achieving performance close to full fine-tuning. LoRA is especially useful for adapting large language models (LLMs) like GPT-2 on downstream tasks with limited resources.

In [74]:
lora_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)

In [75]:
# Print the model artitechture
print(lora_model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)


In [76]:

# TODO: Define your LoRA configuration using LoraConfig class from peft library
#       Apply the LoRA on Conv1D modules (c_attn and c_proj) of GPT2Attention blocks (attn).
#       Set fan_in_fan_out to True
#       Set task_type to CAUSAL_LM
lora_config = LoraConfig(
    r=256,
    lora_alpha=512,
    target_modules=["c_attn", "c_proj"],
    fan_in_fan_out=True,
    task_type="CAUSAL_LM"
)

# # TODO: Wrraped the transformer module of GPT2LMHeadModel with above lora config
# #       using get_peft_model function
lora_model = get_peft_model(lora_model, lora_config)

# TODO: Print number of trainable parameters
lora_model.print_trainable_parameters()

trainable params: 69,206,016 || all params: 424,029,184 || trainable%: 16.3211


In [77]:
# Print the model artitechture and see the changes
print(lora_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 1024)
        (wpe): Embedding(1024, 1024)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-23): 24 x GPT2Block(
            (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=3072, nx=1024)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=256, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=256, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora

In [78]:
# Initialize Trainer
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [79]:
# TODO: Get reserved memory from cuda
gpu_memory_before = torch.cuda.memory_reserved()
# TODO: Train the model using trainer
train_output = trainer.train()
# TODO: Get reserved memory from cuda
gpu_memory_after = torch.cuda.memory_reserved()

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")
print(f"GPU memory used: {(gpu_memory_after - gpu_memory_before) / (1024 ** 3):.4f} GB")

Step,Training Loss


Training time: 89.5366 seconds
GPU memory used: 2606759936.0000 bytes
GPU memory used: 2.4277 GB


In [80]:
# TODO: Evaluate model on eval_dataset
eval_output = trainer.evaluate(eval_dataset=eval_dataset)

print(f"eval_loss = {eval_output['eval_loss']:.4f}")

eval_loss = 1.2210


In [70]:
# Delete the model
del lora_model
del trainer

In [73]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close or less than 1.5Gb)
gc.collect()
torch.cuda.empty_cache()

#### Run LoRA for different rank values

Fine-tune the GPT-2 model with different rank values. (Be sure to change the alpha value according to the rank so that the results are fair.)

Enter the requested items in the table.

Compare the values ​​obtained and explain their differences.

TODO

| Method | Training Time(s) | Training Memory(Gb) | Validation Loss| #Trainable Params(M)|
|:-:|:-:|:-:|:-:|:-:|
| Zero-Shot         |  ... | ...  | 6.9568 | ... |
| Full Fine-Tuning  | 118.5920  | 5.162  | 1.0830 | 124,439,808 |
| Prefix Tuning     | 66.5427  | 1.4395  | 9.4654 | 491,520 |
| Lora rank=4       |  72.0314  | 1.7539  | 4.5581 | 1,081,344 |
| Lora rank=16      | 72.9109  | 1.7832  | 2.8183 | 4,325,376 |
| Lora rank=64      | 75.5601  | 1.9805  | 1.3133 | 17,301,504 |
| Lora rank=256     | 89.5366  | 2.4277  | 1.2210 | 69,206,016 |




TODO:

Your detailed and complete explanation

Zero-shot performs worst with a validation loss of 6.95 since the model isn't adapted to the task. Full fine-tuning achieves the best performance (1.08 loss) but requires the most memory (5.16 GB) and parameters (124M). In contrast, LoRA shows a clear tradeoff between rank and performance: as the rank increases from 4 to 256, validation loss improves (4.56 → 1.22) while parameters and memory grow. LoRA rank 64 offers a strong balance—achieving near full fine-tuning performance (1.31 loss) with far fewer parameters (17M) and lower memory. Rank 4 and 16 are more efficient but less accurate, while rank 256 nearly matches full fine-tuning at higher resource cost.

## Implement LoRA from scratch (30 pt)

In [30]:
custom_lora_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)
print(custom_lora_model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)


In [31]:
from transformers import GPT2LMHeadModel, Conv1D

class LoRALayer(nn.Module):
    def __init__(self, base_layer: Conv1D, rank=8, alpha=16):
        super().__init__()
        self.base_layer = base_layer
        self.rank = rank
        self.alpha = alpha

        # Correctly extract dimensions from Conv1D's weight matrix
        self.in_features = base_layer.weight.shape[0]  # Input dimension (nx)
        self.out_features = base_layer.weight.shape[1]  # Output dimension (nf)

        # Freeze base layer
        for param in base_layer.parameters():
            param.requires_grad = False

        # Initialize LoRA matrices
        self.A = nn.Parameter(torch.randn(rank, self.in_features))  # (rank, in)
        self.B = nn.Parameter(torch.zeros(self.out_features, rank))  # (out, rank)
        nn.init.normal_(self.A, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Original layer output
        base_output = self.base_layer(x)  # (batch, seq_len, out_features)

        # Reshape for matrix operations
        original_shape = x.shape
        x_flat = x.view(-1, original_shape[-1])  # (batch*seq_len, in_features)

        # LoRA adjustment
        lora_output = (x_flat @ self.A.T) @ self.B.T  # (batch*seq_len, out_features)
        lora_output = lora_output.view(*original_shape[:-1], -1)  # Restore shape

        return base_output + lora_output * (self.alpha / self.rank)

In [32]:
# TODO: Freeze the model
for param in custom_lora_model.parameters():
    param.requires_grad = False

# TODO: Loop over list of GPT2Blocks of model and replace the Conv1D
#       modules (c_attn, c_proj) of them with your LoRALayer

for block in custom_lora_model.transformer.h:
    # Replace attention projection (c_attn)
    if isinstance(block.attn.c_attn, Conv1D):
        block.attn.c_attn = LoRALayer(block.attn.c_attn)

    # Replace output projection (c_proj)
    if isinstance(block.attn.c_proj, Conv1D):
        block.attn.c_proj = LoRALayer(block.attn.c_proj)

In [33]:
print(custom_lora_model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): LoRALayer(
            (base_layer): Conv1D(nf=3072, nx=1024)
          )
          (c_proj): LoRALayer(
            (base_layer): Conv1D(nf=1024, nx=1024)
          )
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Lin

In [34]:
# Initialize Trainer
trainer = Trainer(
    model=custom_lora_model,
    args=training_args,
    train_dataset=train_dataset,
)

In [35]:
# TODO: Get reserved memory from cuda
gpu_memory_before = torch.cuda.memory_reserved()
# TODO: Train the model using trainer
train_output = trainer.train()
# TODO: Get reserved memory from cuda
gpu_memory_after = torch.cuda.memory_reserved()

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")
print(f"GPU memory used: {(gpu_memory_after - gpu_memory_before) / (1024 ** 3):.4f} GB")


Step,Training Loss


Training time: 68.2632 seconds
GPU memory used: 1124073472.0000 bytes
GPU memory used: 1.0469 GB


In [36]:
# TODO: Evaluate model on eval_dataset
eval_output = trainer.evaluate(eval_dataset=eval_dataset)

print(f"eval_loss = {eval_output['eval_loss']:.4f}")

eval_loss = 4.6156


In [37]:
# Delete the model
del custom_lora_model
del trainer

In [40]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close or less than 1.5Gb)
gc.collect()
torch.cuda.empty_cache()